In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt 
import seaborn as sns
%matplotlib inline
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import warnings 
warnings.filterwarnings('ignore')
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df_train = pd.read_csv('/kaggle/input/forest-cover-type-prediction/train.csv')
df_test = pd.read_csv('/kaggle/input/forest-cover-type-prediction/test.csv')

In [ ]:
df_train.head()

In [ ]:
df_train.dtypes

There is no categorical data, just numerical.

In [ ]:
pd.set_option('display.max_columns',None)
df_train.describe()

**Inferences**
1. The count is 15120 for each column, which means that no data point is missing.
2. Soil type 7 and 15 are all zeroes so we can remove them.
3. Wilderness_Area and Soil_Type are one hot encoded. Hence, they could be converted back for analysis.
4. We also need to standardize the data as it is not scaled.

In [ ]:
# Removing Soil_type 7 & 15
df_train = df_train.drop(['Soil_Type7','Soil_Type15'],axis=1)
df_test = df_test.drop(['Soil_Type7','Soil_Type15'],axis=1)


**Correlation matrix**

In [ ]:
size = 10 
corrmatrix = df_train.iloc[:,:size].corr()
f,ax = plt.subplots(figsize=(10,8))
sns.heatmap(corrmatrix,vmax=0.8,square = True)

In [ ]:
from sklearn.model_selection import train_test_split
# X = df_train.iloc[:, :-1]
# y = df_train.iloc[:,-1]

X = df_train.drop(['Id'],axis=1)
y = df_train['Cover_Type']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=42)

from sklearn.preprocessing import StandardScaler

features_to_scale = ['Elevation', 'Aspect','Slope','Horizontal_Distance_To_Hydrology','Vertical_Distance_To_Hydrology',
'Horizontal_Distance_To_Roadways','Hillshade_9am','Hillshade_Noon','Hillshade_3pm','Horizontal_Distance_To_Fire_Points']

sc_X = StandardScaler()
X_train[features_to_scale] = sc_X.fit_transform(X_train[features_to_scale])

In [ ]:
X_train.head()

In [ ]:
# Lets try a RandomForestClassifier 
from sklearn.ensemble import RandomForestClassifier 
RFClassifier = RandomForestClassifier(n_estimators=100, random_state=42)
RFClassifier.fit(X_train, y_train)

y_pred = RFClassifier.predict(X_test)

from sklearn.metrics import accuracy_score, confusion_matrix

print(accuracy_score(y_test,y_pred))
# rfclassifier_prob = RFClassifier.predict_proba(X_test)
# confusion_matrix(y_test,y_pred)

In [ ]:
# from xgboost import XGBClassifier 

# xgb = XGBClassifier(learning_rate=0.09,n_estimator=500,max_depth = 30,nthread = 4,objective = 'multi:softprob',subsample=0.75)

# xgb.fit(X_train, y_train)

# y_pred = xgb.predict(X_test)

In [ ]:
from lightgbm import LGBMClassifier 
lgb = LGBMClassifier(learning_rate=0.09,
                    num_leaves = 500,
                     boosting_type='gbdt',
                     objective = 'multiclass',
                     metric = 'multi_logloss',
                     max_depth = 30,
                     subsample=0.75
                    )

lgb.fit(X_train, y_train)

y_pred = lgb.predict(X_test)
print(accuracy_score(y_test,y_pred))

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier 

etc = ExtraTreesClassifier(n_estimators=500,n_jobs=-1,random_state=0)
etc.fit(X_train, y_train)

y_pred = etc.predict(X_test)
print(accuracy_score(y_test, y_pred))

In [ ]:
ids = df_test['Id']
# df_test.drop(['Id'],axis=1,inplace=True)
df_test[features_to_scale] = sc_X.fit_transform(df_test[features_to_scale])
y_result = etc.predict(df_test)

y_result = pd.Series(y_result,name='Cover_Type')
ids = pd.Series(ids, name='Id')
submission = pd.concat([ids,y_result],axis=1)
submission.to_csv('/kaggle/working/submission.csv',index=False)